# Policy Gradient

In [3]:
from mountaincar_utils import test_car, env_mountaincar, display_frames_as_gif, ReplayMemory, goalAchieved

In [4]:
import torch
import torch.nn as nn

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 64), 
            nn.ReLU()
        )
        self.mean_layer = nn.Linear(64, 1)
        self.log_std_layer = nn.Linear(64, 1)

    def forward(self, state):
        x = self.shared(state)
        # Binds mean action to [-1.0, 1.0]
        mean = torch.tanh(self.mean_layer(x)) 
        # Keeps standard deviation positive
        std = torch.exp(torch.clamp(self.log_std_layer(x), -20.0, 2.0)) 
        return mean, std

In [5]:
from torch.distributions.beta import Beta

class BetaPolicyNetwork(nn.Module):
    def __init__(self, state_dim=2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        # Output layers for the two shape parameters
        self.alpha_layer = nn.Linear(64, 1)
        self.beta_layer = nn.Linear(64, 1)

    def forward(self, x):
        features = self.fc(x)
        
        # Softplus ensures alpha and beta are positive. 
        # Adding 1.0 allows the distribution to form concave U-shapes.
        alpha = torch.nn.functional.softplus(self.alpha_layer(features)) + 1.0
        beta = torch.nn.functional.softplus(self.beta_layer(features)) + 1.0
        
        return alpha, beta

In [18]:
from torch.distributions import Normal
import numpy as np

class DQLAgent:
    def __init__(self, env, batch_size=128, epsilon=1.0, epsilon_decay=1.0/3000, epsilon_min=0.1):
        self.env = env

        self.state_dim = 2 # 2 state variables: position & velocity
        self.model = PolicyNetwork(self.state_dim) # output dim=2 for mean and std

        self.epsilon = epsilon #exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_final = epsilon_min
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=0.001)

        self.batch_size = batch_size
        
        self.reset()

    def reset(self):
        self.log_prob = -1.0
        self.action_delta = 0.0
        self.action_value = 0.0

    def act(self, state, train=True):
        if isinstance(state, np.ndarray):
            state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            # state = state * torch.tensor([1.0, 2.0], dtype=torch.float32)
            mean, std = self.model(state)
            dist = Normal(mean.detach(), std.detach())
            self.action_delta = dist.sample().detach().clone().item()

        self.action_value += self.action_delta
        # convert the scalar action to the shape expected by MountainCarContinuous
        # env_input = (self.action * 2.0) - 1.0
        env_input = np.clip(self.action_value, -1.0, 1.0)
        env_input = np.array([env_input], dtype=np.float32)

        return env_input
    
    def update_epsilon(self):
        self.epsilon -= self.epsilon_decay
        if self.epsilon < self.epsilon_final:
            self.epsilon = self.epsilon_final
    
    
    def learn(self, play, alpha=0.9, gamma=0.8):
        def cum_reward(rewards, gamma=0.99):
            cum_rewards = torch.zeros_like(rewards).to(torch.float32)
            for j in range(len(rewards))[::-1]:
                cum_rewards[j] = rewards[j] + gamma * (cum_rewards[j+1] if j+1<len(rewards) else 0)
            eps = np.finfo(np.float32).eps.item()
            cum_rewards = (cum_rewards - cum_rewards.mean()) / (cum_rewards.std() + eps)
            return cum_rewards
        
        # get data
        samples = play.sample()
        
        # get samples batch
        currstates, rewards, actions, states, dones = samples
  
        # compute cumulated reward
        rewards_cum = cum_reward(rewards)

        # compute probabilities
        outputs = self.model(currstates)
        dists = Normal(outputs[0], outputs[1])
        log_probs = dists.log_prob(actions)

        # update network
        loss = (-log_probs * rewards_cum).sum()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.detach().item()
        

In [30]:
# train loop

from collections import deque

epochs = 2000
memory = ReplayMemory()
agent = DQLAgent(env_mountaincar)

scores = []
losses = []
recent_scores = deque(maxlen=100)
frames = []

for e in range(epochs):
    # reset environment
    state, _ = env_mountaincar.reset() # reset environment and get initial state
    agent.reset() # reset agent's action value and action id

    currState = state.copy()
    done = False

    tot_loss = 0
    count = 0
    tot_reward = 0
    rewards = []
    continuing = False
    action = 0.0
    frame_num = 0

    # run an episode
    while not done:
        
       # choose action
        action = agent.act(state)

        # action += np.random.normal(0, 0.3, size=action.shape)  # Add Gaussian noise for exploration

        # take action on env
        state, reward, terminated, truncated, info = env_mountaincar.step(action)
        done = terminated or truncated   

        # # Reward the agent heavily for moving fast and climbing high
        # energy_bonus = (state[1] ** 2) + (state[0] + 0.5)

        # # If it reaches the goal flag (position >= 0.45), give a massive reward
        # reward += energy_bonus   
        # 

        # reward += 0.1 * (0.45 - state[0]) + 0.1 * abs(state[1])  # Reward for position and velocity
        # if done:
        #     reward = 100.0  # Give a large positive reward for reaching the goal
        # if score > 800:
        #     reward = -100  
        #     done = True  # Stop the episode if the score exceeds 900 to prevent infinite loops
        
        # add to replay memory
        memory.add([currState, reward, agent.action_delta, state, done])

        currState = state.copy()

        # update score
        tot_reward += reward
        frame_num += 1

    # train
    loss = agent.learn(memory)

    # clear memory
    memory.clear()
    
    losses = np.append(losses, loss)
    # early stopping if the goal is reached
    recent_scores.append(tot_reward)
    frames.append(frame_num)

    if (e+1)%100 == 0:
        print(f"epoche: {e+1}, reward: {sum(recent_scores)/100}, frames: {frame_num}, loss: {loss:.6f}")

    # early stopping if the goal is reached
    if len(recent_scores) >= 100:
        average_reward = sum(recent_scores) / 100
        if average_reward > 80.:
            print(f"Early stopping at episode {e+1} with average reward: {average_reward:.2f}")
            break

epoche: 100, reward: -38.29779866818224, frames: 999, loss: -0.218453
epoche: 200, reward: -38.21331492657594, frames: 999, loss: -6.697740
epoche: 300, reward: -85.93768850864063, frames: 999, loss: -10.800252
epoche: 400, reward: -78.25895415962117, frames: 999, loss: -25.951712
epoche: 500, reward: -57.434562743690286, frames: 999, loss: -16.495996
epoche: 600, reward: -83.91690322051839, frames: 999, loss: 9.435713
epoche: 700, reward: -70.9596648797114, frames: 999, loss: 5.329014
epoche: 800, reward: -13.432666969785362, frames: 999, loss: -0.570597
epoche: 900, reward: -31.01631968828934, frames: 999, loss: -38.734005
epoche: 1000, reward: -20.602473069641604, frames: 94, loss: 9.590451
epoche: 1100, reward: -40.97090065249846, frames: 999, loss: 57.219456
epoche: 1200, reward: -91.82855544599876, frames: 999, loss: -1.172422
epoche: 1300, reward: -97.6562280914197, frames: 999, loss: 33.004921
epoche: 1400, reward: -72.900303082991, frames: 219, loss: 6.579020
epoche: 1500, rew

In [23]:
# def run_complete_episode_beta(env, policy_net):
#     states, actions, log_probs, rewards = [], [], [], []
    
#     state, info = env.reset()
#     done = False
    
#     while not done:
#         # Scale velocity input state by 15.0 to give the network clear numerical resolution
#         state_scaled = state.copy()
#         state_scaled[1] *= 15.0
#         state_tensor = torch.FloatTensor(state_scaled)
        
#         # 1. Get Beta shape parameters
#         alpha, beta = policy_net(state_tensor)
#         dist = Beta(alpha, beta)
        
#         # 2. Sample from the distribution (returns value in range)
#         raw_action = dist.sample()
        
#         # 3. Calculate the log probability BEFORE scaling
#         log_prob = dist.log_prob(raw_action)
#         log_probs.append(log_prob)
        
#         # 4. Scale the action from [0, 1] to the environment's [-1.0, 1.0] range
#         action_np = (raw_action.detach().numpy() * 2.0) - 1.0
        
#         # 5. Step the environment
#         next_state, reward, terminated, truncated, info = env.step(action_np)
        
#         states.append(state)
#         actions.append(action_np)
#         rewards.append(reward)
        
#         state = next_state
#         done = terminated or truncated
        
#     return {'states': states, 'actions': actions, 'log_probs': log_probs, 'rewards': rewards}

In [33]:
from IPython.display import HTML
frames = test_car(env_mountaincar, 900, agent=agent)
anim = display_frames_as_gif(frames)
HTML(anim.to_jshtml())

84
